# PixArt-Sigma 512: Manifest → T5 Embeddings → Cache Validation

This Colab notebook creates the text-conditioning cache for `jackyckp/efficient-pixart-sigma-lora`. It is the companion to `pixart_clean_latents_colab_english.ipynb`.

It will:

1. Mount Google Drive and read the **existing** `manifest.jsonl` produced by the clean-latent notebook;
2. Preserve the manifest's exact `sample_id` order instead of scanning or sorting the dataset again;
3. Load the T5 encoder and tokenizer bundled with PixArt-Sigma;
4. Encode every caption to `prompt_embeds` with shape `[N, 300, 4096]` and `attention_masks` with shape `[N, 300]`;
5. Cache one empty-prompt embedding for optional classifier-free prompt dropout;
6. Save, reload, and strictly validate the text cache against both the manifest and the clean-latent cache.

The T5 encoder is frozen and used only under `torch.inference_mode()`. The generated cache contains text conditions, not token IDs alone.

> In Colab, select `Runtime → Change runtime type → T4 GPU`, then run every cell from top to bottom. The final FP16 cache for 260 captions is expected to be roughly 610–615 MiB.


In [1]:
#@title 1. Install dependencies (about 1–3 minutes on the first run)
%pip install -q "diffusers==0.39.0" "transformers>=4.46,<6" "accelerate>=1.2" "safetensors>=0.5" "sentencepiece>=0.2" "tqdm>=4.66"


In [2]:
#@title 2. Configure paths, mount Drive, and check the GPU
from google.colab import drive
drive.mount("/content/drive")

import gc
import hashlib
import json
import os
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import T5EncoderModel, T5Tokenizer

assert torch.cuda.is_available(), (
    "No CUDA GPU was detected. Select a T4 GPU from the Runtime menu, "
    "then run this cell again."
)

COMPONENT_MODEL = "PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers"
TRANSFORMER_MODEL = "PixArt-alpha/PixArt-Sigma-XL-2-512-MS"

CLEAN_LATENT_DIR = Path("/content/drive/MyDrive/pixart_sigma/clean_latents_512")
MANIFEST_PATH = CLEAN_LATENT_DIR / "manifest.jsonl"
OUTPUT_DIR = Path("/content/drive/MyDrive/pixart_sigma/t5_embeddings_512")

EXPECTED_NUM_SAMPLES = 260  # Change to 300 if the dataset is expanded; use None to disable
MAX_SEQUENCE_LENGTH = 300   # PixArt-Sigma uses 300 T5 tokens
BATCH_SIZE = 1              # Safest setting for a 16 GiB T4; try 2 only after a successful run
STORAGE_DTYPE = torch.float16
FORCE_REBUILD = False

# This notebook mirrors the direct tokenization used by PixArt training:
# captions are read from the manifest as-is (they were already stripped when written).
CAPTION_PREPROCESSING = "raw_manifest_caption"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU             : {gpu_name} ({vram_gb:.1f} GiB)")
print(f"Manifest        : {MANIFEST_PATH}")
print(f"Text cache dir  : {OUTPUT_DIR}")
print(f"Sequence length : {MAX_SEQUENCE_LENGTH}")


Mounted at /content/drive
GPU             : Tesla T4 (14.6 GiB)
Manifest        : /content/drive/MyDrive/pixart_sigma/clean_latents_512/manifest.jsonl
Text cache dir  : /content/drive/MyDrive/pixart_sigma/t5_embeddings_512
Sequence length : 300


## Input Contract

This notebook deliberately does **not** extract the image archive or rediscover caption files. It consumes the exact manifest written by the clean-latent notebook.

Required manifest fields:

- `sample_id`
- `caption`
- `original_width`
- `original_height`

The manifest order is the shared index for both caches:

```text
image_latent_cache["sample_ids"][i]
    ==
t5_embedding_cache["sample_ids"][i]
```

If this equality fails, training must stop. A valid embedding paired with the wrong image is still incorrect training data.


In [3]:
#@title 3. Load and validate the manifest and matching clean-latent cache
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}\n"
        "Run pixart_clean_latents_colab_english.ipynb first, or update MANIFEST_PATH."
    )

records = []
with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f"Invalid JSON on manifest line {line_number}: {error}") from error
        required = {"sample_id", "caption", "original_width", "original_height"}
        missing = required - set(row)
        if missing:
            raise ValueError(f"Manifest line {line_number} is missing fields: {sorted(missing)}")
        if not isinstance(row["caption"], str) or not row["caption"].strip():
            raise ValueError(f"Empty or invalid caption on manifest line {line_number}")
        records.append(row)

if not records:
    raise RuntimeError("The manifest is empty.")
if EXPECTED_NUM_SAMPLES is not None and len(records) != EXPECTED_NUM_SAMPLES:
    raise ValueError(
        f"Expected {EXPECTED_NUM_SAMPLES} samples, but the manifest contains {len(records)}. "
        "If the dataset was intentionally updated, change EXPECTED_NUM_SAMPLES."
    )

sample_ids = [row["sample_id"] for row in records]
captions = [row["caption"] for row in records]
if len(sample_ids) != len(set(sample_ids)):
    duplicates = sorted({value for value in sample_ids if sample_ids.count(value) > 1})
    raise ValueError(f"Duplicate sample_id values: {duplicates[:10]}")

manifest_fingerprint = hashlib.sha256(
    json.dumps(
        [
            (
                row["sample_id"],
                row["caption"],
                row["original_width"],
                row["original_height"],
            )
            for row in records
        ],
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()[:12]

expected_latent_path = (
    CLEAN_LATENT_DIR
    / f"image_latents_n{len(records)}_res512_{manifest_fingerprint}.pt"
)
if not expected_latent_path.is_file():
    candidates = sorted(CLEAN_LATENT_DIR.glob("image_latents_n*_res512_*.pt"))
    raise FileNotFoundError(
        "The clean-latent cache matching this manifest fingerprint was not found.\n"
        f"Expected: {expected_latent_path}\n"
        f"Other candidates: {[path.name for path in candidates]}"
    )

latent_cache = torch.load(expected_latent_path, map_location="cpu", weights_only=True)
if latent_cache["sample_ids"] != sample_ids:
    raise ValueError("The manifest sample order does not match the clean-latent cache.")
if latent_cache.get("manifest_fingerprint") != manifest_fingerprint:
    raise ValueError("The manifest fingerprint does not match the clean-latent cache.")
if latent_cache.get("transformer_model") != TRANSFORMER_MODEL:
    raise ValueError(
        "The clean-latent cache records a different Transformer model: "
        f"{latent_cache.get('transformer_model')}"
    )

print(f"Samples              : {len(records)}")
print(f"Manifest fingerprint : {manifest_fingerprint}")
print(f"Clean-latent cache   : {expected_latent_path.name}")
print(f"First sample IDs     : {sample_ids[:5]}")
print("\nFirst three captions:")
for index in range(min(3, len(records))):
    print(f"[{sample_ids[index]}] {captions[index]}")

# The large latent tensor is no longer needed after the alignment checks.
del latent_cache
gc.collect()


Samples              : 260
Manifest fingerprint : b9d3c2d1d404
Clean-latent cache   : image_latents_n260_res512_b9d3c2d1d404.pt
First sample IDs     : ['animal/21', 'animal/31', 'animal/38', 'animal/44', 'animal/54']

First three captions:
[animal/21] A serene ink wash painting featuring two small birds, likely thrushes, standing on a snowy ground with sparse grass. The birds, one facing forward and the other slightly turned, have delicate, smooth textures with soft, blended brushstrokes. The background is minimalist with soft, monochromatic washes in shades of gray and white, enhancing the tranquil winter scene. A, Chinese ink wash painting style, Sumi-e
[animal/31] A minimalist ink wash painting featuring two black lizards entwined with purple flowers and green leaves, set against a plain white background. The lizards are drawn with fluid, dynamic lines, emphasizing motion and texture. The flowers add a touch of color and contrast. The artist's signature, "Jiang," is visible in red i

173

## T5 Encoding Settings

PixArt-Sigma uses a frozen `T5EncoderModel` with a maximum text length of 300 tokens. The tokenizer applies:

```text
padding="max_length"
max_length=300
truncation=True
add_special_tokens=True
```

Each caption therefore produces:

- one hidden-state matrix: `[300, 4096]`;
- one attention mask: `[300]`.

Padding positions still have hidden-state values, but the attention mask tells the PixArt Transformer which positions are real caption tokens. Both tensors are required.


In [4]:
#@title 4. Load the PixArt-Sigma tokenizer and frozen T5 encoder
tokenizer = T5Tokenizer.from_pretrained(
    COMPONENT_MODEL,
    subfolder="tokenizer",
)

text_encoder = T5EncoderModel.from_pretrained(
    COMPONENT_MODEL,
    subfolder="text_encoder",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).eval().to("cuda")
text_encoder.requires_grad_(False)

embedding_dim = int(text_encoder.config.d_model)
assert embedding_dim == 4096, embedding_dim
assert text_encoder.dtype == torch.float16

print(f"Tokenizer       : {COMPONENT_MODEL}/tokenizer")
print(f"Text encoder    : {COMPONENT_MODEL}/text_encoder")
print(f"Embedding width : {embedding_dim}")
print(f"Model dtype     : {text_encoder.dtype}")
print(f"Allocated VRAM  : {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")


tokenizer_config.json:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

tokenizer/spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

tokenizer/spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Tokenizer       : PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers/tokenizer
Text encoder    : PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers/text_encoder
Embedding width : 4096
Model dtype     : torch.float16
Allocated VRAM  : 10.75 GiB


In [5]:
#@title 5. Inspect token lengths and truncation risk
untruncated = tokenizer(
    captions,
    padding=False,
    truncation=False,
    add_special_tokens=True,
)
token_lengths = [len(ids) for ids in untruncated["input_ids"]]
truncated_indices = [
    index for index, length in enumerate(token_lengths)
    if length > MAX_SEQUENCE_LENGTH
]

print(f"Minimum token length : {min(token_lengths)}")
print(f"Median token length  : {sorted(token_lengths)[len(token_lengths) // 2]}")
print(f"Maximum token length : {max(token_lengths)}")
print(f"Captions truncated   : {len(truncated_indices)} / {len(captions)}")

if truncated_indices:
    print("\nFirst captions that exceed the limit:")
    for index in truncated_indices[:5]:
        print(
            f"- {sample_ids[index]}: {token_lengths[index]} tokens | "
            f"{captions[index][:180]}"
        )
else:
    print("PASS — no caption exceeds the 300-token PixArt-Sigma limit.")


Minimum token length : 38
Median token length  : 100
Maximum token length : 115
Captions truncated   : 0 / 260
PASS — no caption exceeds the 300-token PixArt-Sigma limit.


In [6]:
#@title 6. Define the batch encoder and run a five-caption smoke test
def encode_caption_batch(batch_captions):
    tokens = tokenizer(
        batch_captions,
        padding="max_length",
        max_length=MAX_SEQUENCE_LENGTH,
        truncation=True,
        add_special_tokens=True,
        return_attention_mask=True,
        return_tensors="pt",
    )
    input_ids = tokens.input_ids.to("cuda")
    attention_mask = tokens.attention_mask.to("cuda")
    with torch.inference_mode():
        prompt_embeds = text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False,
        )[0]
    return (
        prompt_embeds.to(dtype=STORAGE_DTYPE).cpu().contiguous(),
        attention_mask.cpu().to(torch.int64).contiguous(),
    )

smoke_count = min(5, len(captions))
smoke_embeds, smoke_masks = encode_caption_batch(captions[:smoke_count])

assert smoke_embeds.shape == (
    smoke_count,
    MAX_SEQUENCE_LENGTH,
    embedding_dim,
)
assert smoke_masks.shape == (smoke_count, MAX_SEQUENCE_LENGTH)
assert smoke_embeds.dtype == STORAGE_DTYPE
assert smoke_masks.dtype == torch.int64
assert torch.isfinite(smoke_embeds).all()
assert set(torch.unique(smoke_masks).tolist()).issubset({0, 1})
assert (smoke_masks.sum(dim=1) > 0).all()

print("PASS — smoke embedding shape:", tuple(smoke_embeds.shape))
print("PASS — smoke mask shape     :", tuple(smoke_masks.shape))
print(
    "Embedding min/max/mean/std  :",
    smoke_embeds.min().item(),
    smoke_embeds.max().item(),
    smoke_embeds.float().mean().item(),
    smoke_embeds.float().std().item(),
)
print("Real tokens per caption     :", smoke_masks.sum(dim=1).tolist())


PASS — smoke embedding shape: (5, 300, 4096)
PASS — smoke mask shape     : (5, 300)
Embedding min/max/mean/std  : -5.68359375 2.384765625 -0.00042751821456477046 0.11444694548845291
Real tokens per caption     : [96, 100, 70, 105, 115]


## Encode and Save the Full Text Cache

The cache is generated in the manifest's fixed order. Core fields:

- `prompt_embeds`: `[N, 300, 4096]`, CPU `float16`;
- `attention_masks`: `[N, 300]`, CPU `int64`;
- `sample_ids`: identical to the clean-latent cache;
- `empty_prompt_embeds` and `empty_prompt_attention_mask`: one reusable unconditional text condition;
- model, tokenizer, sequence-length, preprocessing, and manifest metadata.

The file is written to a temporary path and then atomically renamed. For 260 samples, expect about 610–615 MiB. This exceeds GitHub's normal 100 MiB per-file limit, so upload the Notebook to GitHub but keep the generated `.pt` cache in Drive or approved large-file storage.


In [7]:
#@title 7. Generate and save all T5 embeddings
cache_path = OUTPUT_DIR / (
    f"t5_embeddings_n{len(records)}_len{MAX_SEQUENCE_LENGTH}_"
    f"fp16_{manifest_fingerprint}.pt"
)
temp_cache_path = cache_path.with_suffix(".tmp")

if cache_path.exists() and not FORCE_REBUILD:
    print(f"Cache already exists; skip encoding: {cache_path}")
else:
    all_prompt_embeds = []
    all_attention_masks = []

    for start in tqdm(
        range(0, len(captions), BATCH_SIZE),
        desc="Encoding captions with T5",
    ):
        batch_captions = captions[start:start + BATCH_SIZE]
        batch_embeds, batch_masks = encode_caption_batch(batch_captions)
        all_prompt_embeds.append(batch_embeds)
        all_attention_masks.append(batch_masks)

    prompt_embeds = torch.cat(all_prompt_embeds, dim=0).contiguous()
    attention_masks = torch.cat(all_attention_masks, dim=0).contiguous()
    empty_prompt_embeds, empty_prompt_attention_mask = encode_caption_batch([""])

    text_cache = {
        "format_version": 1,
        "prompt_embeds": prompt_embeds,
        "attention_masks": attention_masks,
        # Compatibility alias for Diffusers/PixArt naming.
        "prompt_attention_mask": attention_masks,
        "sample_ids": sample_ids,
        "captions": captions,
        "relative_caption_paths": [
            row.get("relative_caption_path") for row in records
        ],
        "num_samples": len(records),
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "embedding_dim": embedding_dim,
        "embedding_dtype": "float16",
        "attention_mask_dtype": "int64",
        "text_condition_kind": "t5_encoder_last_hidden_state",
        "text_encoder_model": COMPONENT_MODEL,
        "text_encoder_subfolder": "text_encoder",
        "tokenizer_model": COMPONENT_MODEL,
        "tokenizer_subfolder": "tokenizer",
        "transformer_model": TRANSFORMER_MODEL,
        "caption_preprocessing": CAPTION_PREPROCESSING,
        "tokenizer_settings": {
            "padding": "max_length",
            "max_length": MAX_SEQUENCE_LENGTH,
            "truncation": True,
            "add_special_tokens": True,
        },
        "empty_prompt_embeds": empty_prompt_embeds,
        "empty_prompt_attention_mask": empty_prompt_attention_mask,
        "manifest_filename": MANIFEST_PATH.name,
        "manifest_fingerprint": manifest_fingerprint,
        "paired_clean_latent_cache": expected_latent_path.name,
    }

    torch.save(text_cache, temp_cache_path)
    os.replace(temp_cache_path, cache_path)

    del all_prompt_embeds, all_attention_masks
    del prompt_embeds, attention_masks
    del empty_prompt_embeds, empty_prompt_attention_mask
    del text_cache
    gc.collect()

print(f"Saved/available: {cache_path}")
print(f"Size           : {cache_path.stat().st_size / 1024**2:.1f} MiB")


Encoding captions with T5:   0%|          | 0/260 [00:00<?, ?it/s]

Saved/available: /content/drive/MyDrive/pixart_sigma/t5_embeddings_512/t5_embeddings_n260_len300_fp16_b9d3c2d1d404.pt
Size           : 612.4 MiB


In [8]:
#@title 8. Reload and strictly validate the text cache
text_cache = torch.load(cache_path, map_location="cpu", weights_only=True)

prompt_embeds = text_cache["prompt_embeds"]
attention_masks = text_cache["attention_masks"]

assert prompt_embeds.shape == (
    len(records),
    MAX_SEQUENCE_LENGTH,
    embedding_dim,
), prompt_embeds.shape
assert attention_masks.shape == (
    len(records),
    MAX_SEQUENCE_LENGTH,
), attention_masks.shape
assert prompt_embeds.dtype == STORAGE_DTYPE
assert attention_masks.dtype == torch.int64
assert torch.isfinite(prompt_embeds).all()
assert set(torch.unique(attention_masks).tolist()).issubset({0, 1})
assert (attention_masks.sum(dim=1) > 0).all()
assert text_cache["sample_ids"] == sample_ids
assert text_cache["captions"] == captions
assert text_cache["manifest_fingerprint"] == manifest_fingerprint
assert text_cache["transformer_model"] == TRANSFORMER_MODEL
assert text_cache["max_sequence_length"] == MAX_SEQUENCE_LENGTH
assert text_cache["embedding_dim"] == embedding_dim
assert text_cache["paired_clean_latent_cache"] == expected_latent_path.name
assert torch.equal(text_cache["prompt_attention_mask"], attention_masks)

# Re-encode one caption with the same batch size used for the full cache.
# This avoids false failures caused by small FP16 kernel differences across batch sizes.
reference_embed, reference_mask = encode_caption_batch(captions[:1])
torch.testing.assert_close(
    prompt_embeds[:1],
    reference_embed,
    rtol=2e-3,
    atol=2e-3,
)
assert torch.equal(attention_masks[:1], reference_mask)

empty_embeds = text_cache["empty_prompt_embeds"]
empty_mask = text_cache["empty_prompt_attention_mask"]
assert empty_embeds.shape == (1, MAX_SEQUENCE_LENGTH, embedding_dim)
assert empty_mask.shape == (1, MAX_SEQUENCE_LENGTH)
assert torch.isfinite(empty_embeds).all()

latent_cache = torch.load(
    expected_latent_path,
    map_location="cpu",
    weights_only=True,
)
assert latent_cache["sample_ids"] == text_cache["sample_ids"]
assert latent_cache["manifest_fingerprint"] == text_cache["manifest_fingerprint"]
assert latent_cache["transformer_model"] == text_cache["transformer_model"]
del latent_cache

summary = {
    "status": "PASS",
    "cache_file": cache_path.name,
    "manifest_file": MANIFEST_PATH.name,
    "paired_clean_latent_cache": expected_latent_path.name,
    "num_samples": len(records),
    "prompt_embeds_shape": list(prompt_embeds.shape),
    "prompt_embeds_dtype": str(prompt_embeds.dtype),
    "attention_masks_shape": list(attention_masks.shape),
    "attention_masks_dtype": str(attention_masks.dtype),
    "all_finite": bool(torch.isfinite(prompt_embeds).all()),
    "embedding_min": float(prompt_embeds.min()),
    "embedding_max": float(prompt_embeds.max()),
    "embedding_mean": float(prompt_embeds.float().mean()),
    "embedding_std": float(prompt_embeds.float().std()),
    "minimum_real_tokens": int(attention_masks.sum(dim=1).min()),
    "maximum_real_tokens": int(attention_masks.sum(dim=1).max()),
    "captions_truncated": len(truncated_indices),
    "manifest_fingerprint": manifest_fingerprint,
    "text_encoder_model": COMPONENT_MODEL,
    "transformer_model": TRANSFORMER_MODEL,
}

summary_path = OUTPUT_DIR / "validation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("\nPASS — T5 embedding cache is aligned and ready for PixArt-Sigma LoRA training.")
print(f"Text cache      : {cache_path}")
print(f"Latent cache    : {expected_latent_path}")
print(f"Validation JSON : {summary_path}")


{
  "status": "PASS",
  "cache_file": "t5_embeddings_n260_len300_fp16_b9d3c2d1d404.pt",
  "manifest_file": "manifest.jsonl",
  "paired_clean_latent_cache": "image_latents_n260_res512_b9d3c2d1d404.pt",
  "num_samples": 260,
  "prompt_embeds_shape": [
    260,
    300,
    4096
  ],
  "prompt_embeds_dtype": "torch.float16",
  "attention_masks_shape": [
    260,
    300
  ],
  "attention_masks_dtype": "torch.int64",
  "all_finite": true,
  "embedding_min": -6.08203125,
  "embedding_max": 2.390625,
  "embedding_mean": -0.0008960929117165506,
  "embedding_std": 0.11071175336837769,
  "minimum_real_tokens": 38,
  "maximum_real_tokens": 115,
  "captions_truncated": 0,
  "manifest_fingerprint": "b9d3c2d1d404",
  "text_encoder_model": "PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers",
  "transformer_model": "PixArt-alpha/PixArt-Sigma-XL-2-512-MS"
}

PASS — T5 embedding cache is aligned and ready for PixArt-Sigma LoRA training.
Text cache      : /content/drive/MyDrive/pixart_sigma/t5_embeddings_5

## Loading Both Caches During Training

```python
image_cache = torch.load(IMAGE_CACHE_PATH, map_location="cpu", weights_only=True)
text_cache = torch.load(TEXT_CACHE_PATH, map_location="cpu", weights_only=True)

assert image_cache["sample_ids"] == text_cache["sample_ids"]
assert image_cache["manifest_fingerprint"] == text_cache["manifest_fingerprint"]
assert image_cache["transformer_model"] == text_cache["transformer_model"]

clean_latents = image_cache["latents"]                 # [N, 4, 64, 64]
prompt_embeds = text_cache["prompt_embeds"]             # [N, 300, 4096]
prompt_attention_mask = text_cache["attention_masks"]   # [N, 300]
```

A training sample at index `i` must take all three tensors from the same index. Shuffle indices, not the caches independently.

If the training loop uses caption dropout for classifier-free guidance, replace the selected prompt condition with the cached empty-prompt tensors:

```python
if drop_caption:
    batch_prompt_embeds = text_cache["empty_prompt_embeds"].expand(batch_size, -1, -1)
    batch_prompt_attention_mask = text_cache["empty_prompt_attention_mask"].expand(batch_size, -1)
```

Random diffusion noise and timesteps are still created dynamically from `clean_latents` inside each training step.
